In [3]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

# sys.path.insert(0, "~/Florian-Q/maraicherbio-prediction/notebooks/")

In [5]:
import utils
import utils_series
import pandas as pd
import matplotlib.pyplot as plt

In [6]:
# --- Boucle : df_train / df_test pour chaque produit (split adaptatif, date fin commune) ---
df_all = utils.charger_dataframe()

# Définir la date de fin commune pour TOUS les splits
utils_series.GLOBAL_TEST_END_DATE = df_all['created'].max()
print(f'Date de fin commune : {utils_series.GLOBAL_TEST_END_DATE.date()}\n')

modeles = sorted(df_all['model'].unique())
print(f'{len(modeles)} produits à traiter\n')

dict_train = {}
dict_test = {}
skipped = []

for modele in modeles:
    try:
        df_produit = utils.charger_dataframe(modele)
        df_model = utils_series.complete_weekly_dataframe(df_produit, 'created', 'quantite_y')
        train, test = utils_series.split_adaptive_seasonal(df_model, test_pct=0.20)
        dict_train[modele] = train
        dict_test[modele] = test
    except ValueError as e:
        skipped.append(modele)
        continue

print(f'\nTerminé : {len(dict_train)} produits prêts  |  {len(skipped)} ignorés')
if skipped:
    print(f'Ignorés : {skipped}')

# Vérification
test_ends = [t.index.max().date() for t in dict_test.values()]
print(f'Toutes les fins de test = {test_ends[0]} ?  {len(set(test_ends)) == 1}')

Date de fin commune : 2026-05-27

100 produits à traiter

Train : 2014-06-08  →  2024-05-26  (521 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 12.0 ans (17%)
Train : 2014-07-06  →  2024-05-26  (517 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 11.9 ans (17%)
Train : 2014-06-29  →  2024-05-26  (518 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 11.9 ans (17%)
Train : 2015-01-11  →  2024-05-26  (490 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 11.4 ans (18%)
Train : 2018-05-06  →  2024-05-26  (317 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 8.1 ans (25%)
Train : 2014-01-05  →  2024-05-26  (543 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 12.4 ans (16%)
Train : 2014-02-23  →  2024-05-26  (536 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an

In [ ]:
# ============================================================
# XGBOOST v1 : lag(y_t-1) + semaine ISO + split_adaptive + seasonal_metrics
# ============================================================
import numpy as np
import pandas as pd
import itertools
import model_xgboost

# ── Hyperparamètres fixes ─────────────────────────────────────────────────────
BEST_PARAMS: dict = {
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "objective"        : "reg:squarederror",
    "random_state"     : 42,
    "verbosity"        : 0,
}

# ── Grid Search ───────────────────────────────────────────────────────────────
TUNING_GRID: dict = {
    "n_estimators"  : [100, 300],
    "max_depth"     : [3, 5],
    "learning_rate" : [0.05, 0.1],
}

ALL_COMBOS = list(itertools.product(
    TUNING_GRID["n_estimators"],
    TUNING_GRID["max_depth"],
    TUNING_GRID["learning_rate"],
))
# 2 × 2 × 2 = 8 combinaisons par produit


# ── MODE ÉVALUATION (remplace la boucle XGBoost v1) ──────────────────────────
results = []

for modele in dict_train.keys():
    y_train = dict_train[modele]['quantite_y']
    y_test  = dict_test[modele]['quantite_y']

    best_metric  = np.inf
    best_nest = None
    best_dep  = None
    best_lr   = None

    for n_est, depth, lr in ALL_COMBOS:
        try:
            metrics = model_xgboost.run_xgboost(n_est, depth, lr, y_train, y_test)
            if metrics["MAPE"] < best_metric:
                best_metric     = metrics["MAPE"]
                best_nest    = n_est
                best_dep     = depth
                best_lr      = lr
        except Exception as e:
            print(f"[WARN] {modele} | n_est={n_est} depth={depth} lr={lr} → {e}")
            continue

    if best_metric is None:
        continue

    results.append({
        'produit'    : modele,
        'train_debut': y_train.index.min().strftime('%Y-%m'),
        'best_nest'  : best_nest,
        'best_depth' : best_dep,
        'best_lr'    : best_lr,
        'MAPE'       : round(best_metric,      1),
    })

# ── MODE PRÉDICTION (exemple sur un produit) ──────────────────────────────────
# df_pred = run_xgboost(100, 3, 0.05, dict_train["mon_produit"])
# print(df_pred)

In [12]:
df_xgb = pd.DataFrame(results)
df_xgb

,produit,train_debut,test_fin,train_sem,test_sem,best_nest,best_depth,best_lr,pct_zeros,MAE_in,MAE_out,MAE_all,MAPE,sMAPE_all
0,Ail,2014-06,2026-05,521,104,100,5,0.05,91.3,0.93,0.06,0.14,84.2,26.5
1,Aubergine noire,2014-07,2026-05,517,104,100,3,0.05,68.3,2.95,0.35,1.17,50.2,28.5
2,Basilic,2014-06,2026-05,518,105,100,3,0.05,63.8,3.15,0.11,1.21,109.8,117.8
3,Betterave Jaune,2015-01,2026-05,490,104,100,3,0.05,68.3,0.61,0.14,0.28,104.2,68.9
4,Betterave botte,2018-05,2026-05,317,105,300,3,0.10,64.8,2.05,0.54,1.07,92.8,108.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,patate douce,2020-10,2026-05,243,52,100,3,0.05,73.1,3.24,0.20,1.02,49.7,23.4
80,pomme dalinsweet,2014-04,2026-05,529,104,100,5,0.10,90.4,2.86,0.33,0.57,96.6,58.3
81,radis daikon,2019-10,2026-05,293,52,100,3,0.05,80.8,0.67,0.08,0.19,51.5,47.7
82,tomate ancienne,2019-08,2026-05,304,52,100,5,0.05,63.5,2.84,0.06,1.08,43.3,22.7


In [13]:
df_xgb['MAPE'].mean()

np.float64(73.00714285714287)

In [ ]:
df_xgb.to_csv('../data/XGboost_metrics.csv', index=False)